<a href="https://colab.research.google.com/github/Coobaltoo13/milling-tool-life-fusion/blob/main/08_tsfresh_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Seleccionar solo las features más importantes antes de entrenar

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# ===== CARGAR DATASET UNIFICADO =====
df = pd.read_csv('/content/gdrive/MyDrive/features_tsfresh_completo.csv', index_col=0)
print(f"Dataset: {df.shape}")

# ===== SEPARAR X e y =====
# Excluir etiqueta y columna 'archivo'
columnas_excluir = ['CycleToFailureNormalized', 'archivo']
X = df.drop(columns=[c for c in columnas_excluir if c in df.columns])
y = df['CycleToFailureNormalized'].values

print(f"X: {X.shape}")
print(f"y: {y.shape}")

# ===== LIMPIAR: eliminar features con NaN o infinitos =====
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# ===== DIVIDIR TRAIN/VAL/TEST (60/20/20) =====
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"\nTrain: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

# ===== SELECCIONAR FEATURES SOLO CON TRAIN =====
# Top 500 features por correlación con la etiqueta
K_FEATURES = 500
selector = SelectKBest(score_func=f_regression, k=K_FEATURES)
selector.fit(X_train, y_train)

columnas_seleccionadas = X.columns[selector.get_support()].tolist()
print(f"\nFeatures seleccionadas: {len(columnas_seleccionadas)}")

# Aplicar selección
X_train_sel = X_train[columnas_seleccionadas]
X_val_sel = X_val[columnas_seleccionadas]
X_test_sel = X_test[columnas_seleccionadas]

# ===== NORMALIZAR =====
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_val_scaled = scaler.transform(X_val_sel)
X_test_scaled = scaler.transform(X_test_sel)

print(f"Train normalizado: {X_train_scaled.shape}")
print(f"Val normalizado: {X_val_scaled.shape}")
print(f"Test normalizado: {X_test_scaled.shape}")

# Guardar para siguientes pasos
np.save('/content/gdrive/MyDrive/X_train_tsfresh.npy', X_train_scaled)
np.save('/content/gdrive/MyDrive/X_val_tsfresh.npy', X_val_scaled)
np.save('/content/gdrive/MyDrive/X_test_tsfresh.npy', X_test_scaled)
np.save('/content/gdrive/MyDrive/y_train_tsfresh.npy', y_train)
np.save('/content/gdrive/MyDrive/y_val_tsfresh.npy', y_val)
np.save('/content/gdrive/MyDrive/y_test_tsfresh.npy', y_test)

print("\n💾 Datos guardados en Drive")

Dataset: (968, 10729)
X: (968, 10727)
y: (968,)

Train: (580, 10727)
Val: (194, 10727)
Test: (194, 10727)

Features seleccionadas: 500
Train normalizado: (580, 500)
Val normalizado: (194, 500)
Test normalizado: (194, 500)

💾 Datos guardados en Drive


Entrenar XGBoost

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import numpy as np

# ===== CARGAR DATOS =====
X_train = np.load('/content/gdrive/MyDrive/X_train_tsfresh.npy')
X_val = np.load('/content/gdrive/MyDrive/X_val_tsfresh.npy')
X_test = np.load('/content/gdrive/MyDrive/X_test_tsfresh.npy')
y_train = np.load('/content/gdrive/MyDrive/y_train_tsfresh.npy')
y_val = np.load('/content/gdrive/MyDrive/y_val_tsfresh.npy')
y_test = np.load('/content/gdrive/MyDrive/y_test_tsfresh.npy')

# ===== FUNCIÓN DE MÉTRICAS =====
def calcular_metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    y_true_class = (y_true >= 0.5).astype(int)
    y_pred_class = (y_pred >= 0.5).astype(int)
    f1 = f1_score(y_true_class, y_pred_class, zero_division=0)
    precision = precision_score(y_true_class, y_pred_class, zero_division=0)
    recall = recall_score(y_true_class, y_pred_class, zero_division=0)
    try:
        auc = roc_auc_score(y_true_class, y_pred)
    except:
        auc = np.nan
    return rmse, mae, r2, f1, precision, recall, auc

# ===== ENTRENAR XGBOOST =====
print("Entrenando XGBoost con features de tsfresh...")
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                   random_state=42, verbosity=0)
xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False)

y_pred_xgb = xgb.predict(X_test)
rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_test, y_pred_xgb)

print("\n" + "=" * 50)
print("XGBOOST CON FEATURES DE TSFRESH (500 features)")
print("=" * 50)
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")
print(f"R²:        {r2:.4f}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"AUC:       {auc:.4f}")

# Guardar métricas de XGBoost
rmse_xgb, mae_xgb, r2_xgb, f1_xgb, precision_xgb, recall_xgb, auc_xgb = rmse, mae, r2, f1, precision, recall, auc

Entrenando XGBoost con features de tsfresh...

XGBOOST CON FEATURES DE TSFRESH (500 features)
RMSE:      0.1090
MAE:       0.0786
R²:        0.8527
F1:        0.9016
Precision: 0.8878
Recall:    0.9158
AUC:       0.9709


Entrenar LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ===== REFORMAR PARA LSTM =====
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val_lstm = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print(f"Train LSTM: {X_train_lstm.shape}")
print(f"Val LSTM: {X_val_lstm.shape}")
print(f"Test LSTM: {X_test_lstm.shape}")

# ===== DEFINIR LSTM =====
model = models.Sequential([
    layers.Input(shape=(1, X_train.shape[1])),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# ===== ENTRENAR =====
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_lstm, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_lstm, y_val),
    callbacks=[early_stop],
    verbose=1
)

# ===== EVALUAR =====
y_pred_lstm = model.predict(X_test_lstm, verbose=0).flatten()
rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_test, y_pred_lstm)

print("\n" + "=" * 50)
print("LSTM CON FEATURES DE TSFRESH (500 features)")
print("=" * 50)
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")
print(f"R²:        {r2:.4f}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"AUC:       {auc:.4f}")

# Guardar métricas de LSTM
rmse_lstm, mae_lstm, r2_lstm, f1_lstm, precision_lstm, recall_lstm, auc_lstm = rmse, mae, r2, f1, precision, recall, auc

Train LSTM: (580, 1, 500)
Val LSTM: (194, 1, 500)
Test LSTM: (194, 1, 500)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │        68,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,769 (268.63 KB)

 Trainable params: 68,769 (268.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - loss: 0.2018 - mae: 0.3408 - val_loss: 0.0561 - val_mae: 0.1821
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0334 - mae: 0.1419 - val_loss: 0.0339 - val_mae: 0.1464
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0213 - mae: 0.1111 - val_loss: 0.0257 - val_mae: 0.1261
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0155 - mae: 0.0947 - val_loss: 0.0207 - val_mae: 0.1109
Epoch 5/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0117 - mae: 0.0813 - val_loss: 0.0197 - val_mae: 0.1078
Epoch 6/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0094 - mae: 0.0736 - val_loss: 0.0172 - val_mae: 0.1000
Epoch 7/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0075 - mae: 0.0645 - val_loss: 0.0148 - val_mae: 0.0916
Epoch 8/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0064 - mae: 0.0592 - val_loss: 0.0144 - val_mae: 0.0883
Epoch 9/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.00


LSTM CON FEATURES DE TSFRESH (500 features)
RMSE:      0.1247
MAE:       0.0807
R²:        0.8072
F1:        0.9215
Precision: 0.9167
Recall:    0.9263
AUC:       0.9634


Comparacion Final

In [ ]:
import pandas as pd

# ===== COMPARACIÓN FINAL =====
comparacion = pd.DataFrame([
    {'modelo': 'XGBoost (120 features originales)', 'R2': 0.8471, 'RMSE': 0.1111, 'MAE': 0.0745, 'F1': 0.9053, 'AUC': 0.9761},
    {'modelo': 'XGBoost (500 features tsfresh)',    'R2': 0.8527, 'RMSE': 0.1090, 'MAE': 0.0786, 'F1': 0.9016, 'AUC': 0.9709},
    {'modelo': 'LSTM (120 features originales)',    'R2': 0.6805, 'RMSE': 0.1605, 'MAE': 0.1133, 'F1': 0.8772, 'AUC': 0.9431},
    {'modelo': 'LSTM (500 features tsfresh)',       'R2': 0.8228, 'RMSE': 0.1195, 'MAE': 0.0817, 'F1': 0.9053, 'AUC': 0.9674},
])

print("\n" + "=" * 80)
print("COMPARACIÓN FINAL: 120 FEATURES ORIGINALES vs 500 FEATURES TSFRESH")
print("=" * 80)
print(comparacion.to_string(index=False))

# Guardar
comparacion.to_csv('/content/gdrive/MyDrive/comparacion_final.csv', index=False)
print("\n💾 Guardado en Drive: comparacion_final.csv")


COMPARACIÓN FINAL: 120 FEATURES ORIGINALES vs 500 FEATURES TSFRESH
                           modelo     R2   RMSE    MAE     F1    AUC
XGBoost (120 features originales) 0.8471 0.1111 0.0745 0.9053 0.9761
   XGBoost (500 features tsfresh) 0.8527 0.1090 0.0786 0.9016 0.9709
   LSTM (120 features originales) 0.6805 0.1605 0.1133 0.8772 0.9431
      LSTM (500 features tsfresh) 0.8228 0.1195 0.0817 0.9053 0.9674

💾 Guardado en Drive: comparacion_final.csv
